# Ferrous + xarray — Mediterranean air temperature

This notebook fetches a tiny slice of CMIP6 surface air temperature using
[Ferrous](https://github.com/tham-le/ferrous), opens it with xarray, and
plots the first-month snapshot and the annual-mean trend.

The full file is **74 MB**; the slice is **~77 KB**. Everything below
runs against that 77 KB copy on disk.

In [ ]:
# Fetch the slice with Ferrous.
# Adjust --lat-deg / --lon-deg / --time-iso for your region and period.
import subprocess, shutil, os
ferrous = shutil.which("ferrous") or "target/release/ferrous"
out_path = "/tmp/ferrous_tas_med.nc"
if not os.path.exists(out_path):
    subprocess.run([
        ferrous, "get",
        "--dataset-id",
        "CMIP6.ScenarioMIP.CNRM-CERFACS.CNRM-CM6-1.ssp245.r1i1p1f2.Amon.tas.gr.v20190219|esgf.ceda.ac.uk",
        "--variable", "tas",
        "--time-iso", "2020:2025",
        "--lat-deg", "30:46",
        "--lon-deg", "0:30",
        "--out", out_path,
    ], check=True)
out_path

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

ds = xr.open_dataset(out_path)
ds

## First-month snapshot

The time axis is monthly, starting in January 2020. Pull out the first
step and plot it as a filled contour.

In [ ]:
tas = ds["tas"]
tas_c = tas - 273.15  # Kelvin -> Celsius
fig, ax = plt.subplots(figsize=(8, 4))
tas_c.isel(time=0).plot.contourf(ax=ax, levels=12, cmap="RdYlBu_r")
ax.set_title("Mediterranean surface air temperature, Jan 2020 (°C)")
plt.tight_layout()

## Annual-mean temperature trend

Collapse time into yearly bins, then average across the spatial window to
get a single number per year. `coarsen(time=12)` groups 12 months at a
time. The result is 6 data points (one per year, 2020-2025).

In [ ]:
valid = tas_c.where(np.abs(tas_c) < 1e10)

# 6 years x 12 months -> 6 annual points averaged over the region.
annual = (
    valid.coarsen(time=12)
    .mean()
    .mean(dim=[d for d in tas.dims if d != "time"])
)

fig, ax = plt.subplots(figsize=(6, 3))
years = np.arange(2020, 2020 + annual.sizes["time"])
ax.plot(years, annual.values, marker="o")
ax.set(xlabel="year", ylabel="Mediterranean mean tas (°C)",
       title="Annual-mean surface air temperature, 2020-2025")
ax.grid(True, alpha=0.3)
plt.tight_layout()

## That's it

The whole notebook runs against 77 KB of local data. No ESGF login, no
4 GB download, no intake-esm dependency stack — just `ferrous get`
once, then vanilla xarray.

A second run of the notebook hits the local Ferrous cache and transfers
zero bytes from ESGF.